# 1. Context

This notebook does adhoc experimentation with Post OCR Corrections

# 2. Imports

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
from dotenv import load_dotenv

In [ ]:
import torch

In [ ]:
import os

# 3. Model Exploration

## 3.1. Gemma 3

### 3.1.1. Gemma-3 270M 

In [ ]:
model_id = "google/gemma-3-1b-it"

In [ ]:
# Check for CUDA (NVIDIA GPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA GPU.")
# Check for MPS (Apple Silicon Mac)
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon MPS.")
else:
    device = torch.device("cpu")
    print("Using CPU.")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map=device)

In [ ]:
OCR_CORRECTOR_PROMPT = """You are a Post OCR Corrector Model. 
You will be provided with text to be corrected in {text} placeholder.
Rectify any error present and provide corrected output in {corrected_text} placeholder in same language as input text"""

In [ ]:
text = "अतः आप ज्येष्ठ मास की शुक्ल पक्ष की निर्जुला नाम की एक ही एकादशी का व्रत करो और तुम्हें वर्ष की समस्त एकादशियों का फल प्राप्त होगा"

In [ ]:
messages = [
    {"role": "user", "content": f"{OCR_CORRECTOR_PROMPT}\n\ntext: {text}"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

# outputs = model.generate(**inputs, max_new_tokens=100)

In [ ]:
inputs

In [ ]:
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [ ]:
class postCorrectorLLM():
    """Class For OCR Post Correction Using a LLM"""

    def __init__(self, model_id: str):
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(model_id, device_map="cuda")
        self.model_id = model_id
        self.correction_prompt = None
